# **PROBLEM 1**

In [2]:
import pandas as pd
file_path = '/content/PROBLEM1.csv'
df = pd.read_csv(file_path)
print(df)

   Height  Weight  Age  Grip strength Frailty
0    65.8     112   30             30       N
1    71.5     136   19             31       N
2    69.4     153   45             29       N
3    68.2     142   22             28       Y
4    67.8     144   29             24       Y
5    68.7     123   50             26       N
6    69.8     141   51             22       Y
7    70.1     136   23             20       Y
8    67.9     112   17             19       N
9    66.8     120   39             31       N


In [3]:
# Strip column names and drop rows with missing critical values
df.columns = df.columns.str.strip()
df = df.dropna(subset=["Age", "Height", "Weight", "Frailty"])

# Fill missing Grip strength with median if the column exists
if "Grip strength" in df.columns:
    df["Grip strength"] = df["Grip strength"].fillna(df["Grip strength"].median())

# Save cleaned CSV
df.to_csv('/content/PROBLEM1_clean.csv', index=False)
print("Cleaned data saved as PROBLEM1_clean.csv")

Cleaned data saved as PROBLEM1_clean.csv


**a) Unit standardization**

In [5]:
df["Height_m"] = (df["Height"] * 0.0254).round(2)     # Convert inches → meters
df["Weight_kg"] = (df["Weight"] * 0.45359237).round(2)  # Convert pounds → kg
print(df)

   Height  Weight  Age  Grip strength Frailty  Height_m  Weight_kg
0    65.8     112   30             30       N      1.67      50.80
1    71.5     136   19             31       N      1.82      61.69
2    69.4     153   45             29       N      1.76      69.40
3    68.2     142   22             28       Y      1.73      64.41
4    67.8     144   29             24       Y      1.72      65.32
5    68.7     123   50             26       N      1.74      55.79
6    69.8     141   51             22       Y      1.77      63.96
7    70.1     136   23             20       Y      1.78      61.69
8    67.9     112   17             19       N      1.72      50.80
9    66.8     120   39             31       N      1.70      54.43


**b) Feature engineering**

In [6]:
df["BMI"] = (df["Weight_kg"] / (df["Height_m"] ** 2)).round(2)
def age_group(age):
    if age < 30:
        return "<30"
    elif 30 <= age <= 45:
        return "30–45"
    elif 46 <= age <= 60:
        return "46–60"
    else:
        return ">60"

df["AgeGroup"] = df["Age"].apply(age_group)
print(df)

   Height  Weight  Age  Grip strength Frailty  Height_m  Weight_kg    BMI  \
0    65.8     112   30             30       N      1.67      50.80  18.22   
1    71.5     136   19             31       N      1.82      61.69  18.62   
2    69.4     153   45             29       N      1.76      69.40  22.40   
3    68.2     142   22             28       Y      1.73      64.41  21.52   
4    67.8     144   29             24       Y      1.72      65.32  22.08   
5    68.7     123   50             26       N      1.74      55.79  18.43   
6    69.8     141   51             22       Y      1.77      63.96  20.42   
7    70.1     136   23             20       Y      1.78      61.69  19.47   
8    67.9     112   17             19       N      1.72      50.80  17.17   
9    66.8     120   39             31       N      1.70      54.43  18.83   

  AgeGroup  
0    30–45  
1      <30  
2    30–45  
3      <30  
4      <30  
5    46–60  
6    46–60  
7      <30  
8      <30  
9    30–45  


**c) Categorical → numeric encoding**

In [7]:
df["Frailty_binary"] = df["Frailty"].map({"Y": 1, "N": 0}).astype("int8")


df[["Frailty", "Frailty_binary"]]

,Frailty,Frailty_binary
0,N,0
1,N,0
2,N,0
3,Y,1
4,Y,1
5,N,0
6,Y,1
7,Y,1
8,N,0
9,N,0


In [8]:
df["BMI"] = (df["Weight_kg"] / (df["Height_m"] ** 2)).round(2)
def age_group(age):
    if age < 30:
        return "<30"
    elif 30 <= age <= 45:
        return "30–45"
    elif 46 <= age <= 60:
        return "46–60"
    else:
        return ">60"

df["AgeGroup"] = df["Age"].apply(age_group)

df = pd.get_dummies(df, columns=["AgeGroup"], prefix="AgeGroup")
print(df)

   Height  Weight  Age  Grip strength Frailty  Height_m  Weight_kg    BMI  \
0    65.8     112   30             30       N      1.67      50.80  18.22   
1    71.5     136   19             31       N      1.82      61.69  18.62   
2    69.4     153   45             29       N      1.76      69.40  22.40   
3    68.2     142   22             28       Y      1.73      64.41  21.52   
4    67.8     144   29             24       Y      1.72      65.32  22.08   
5    68.7     123   50             26       N      1.74      55.79  18.43   
6    69.8     141   51             22       Y      1.77      63.96  20.42   
7    70.1     136   23             20       Y      1.78      61.69  19.47   
8    67.9     112   17             19       N      1.72      50.80  17.17   
9    66.8     120   39             31       N      1.70      54.43  18.83   

   Frailty_binary  AgeGroup_30–45  AgeGroup_46–60  AgeGroup_<30  
0               0            True           False         False  
1               0   

**d) EDA & Reporting**

In [9]:
import os


numeric_cols = df.select_dtypes(include=["number"]).columns


summary = df[numeric_cols].agg(["mean", "median", "std"]).round(2)


os.makedirs("reports", exist_ok=True)


with open("reports/findings.md", "w") as f:
    f.write("# Summary Statistics\n\n")
    f.write(summary.to_markdown())
print(summary)
summary


        Height  Weight    Age  Grip strength  Height_m  Weight_kg    BMI  \
mean     68.60  131.90  32.50          26.00      1.74      59.83  19.72   
median   68.45  136.00  29.50          27.00      1.74      61.69  19.15   
std       1.67   14.23  12.86           4.52      0.04       6.46   1.79   

        Frailty_binary  
mean              0.40  
median            0.00  
std               0.52  


,Height,Weight,Age,Grip strength,Height_m,Weight_kg,BMI,Frailty_binary
mean,68.60,131.90,32.50,26.00,1.74,59.83,19.72,0.40
median,68.45,136.00,29.50,27.00,1.74,61.69,19.15,0.00
std,1.67,14.23,12.86,4.52,0.04,6.46,1.79,0.52


In [10]:
df.columns = df.columns.str.strip()


correlation = df["Grip strength"].corr(df["Frailty_binary"])

print(f"Correlation between Grip strength and Frailty_binary: {correlation:.2f}")



Correlation between Grip strength and Frailty_binary: -0.48
